####DAY 11 (02/03/26) – Time Travel & Data Recovery
####🏗️ Architecture & Strategy
Welcome to Day 11! Today we are tackling the exact scenario that keeps Data Engineers awake at 3 AM: Data Corruption. What happens if a bad pipeline run accidentally appends millions of duplicate or malformed rows into your production table?

Before Delta Lake, rolling back a massive data warehouse meant restoring from yesterday's rigid backups—a process that took hours. Today, we are leveraging Delta Lake Time Travel .

####Our Senior-Level Strategy:

* **The Sandbox**: We will create a small, temporary demo_time_travel_events table so we can safely simulate a catastrophic data pipeline failure without harming our main project tables.

* **The "Oops" Moment**: We will intentionally append bad records to this table to create a new, corrupted version (Version 1).

* **Delta Transaction Log**: We will use DESCRIBE HISTORY to read the underlying JSON transaction log and prove that Delta Lake tracks every single ACID transaction.

* **Zero-Copy Time Travel**: We will use PySpark's .option("versionAsOf", 0) to query the data exactly as it looked before our mistake. Delta does this instantly without copying data; it just reads the older Parquet files referenced in the transaction log.

* **The 3 AM Lifesaver (Rollback)**: We will execute a RESTORE command to instantly revert the production table back to a healthy state.

###Create the Base Table (Version 0)
Let's set up a safe, isolated table and establish our baseline "healthy" data.

In [0]:
from pyspark.sql import functions as F

# 1. Environment Setup
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

demo_table = "demo_time_travel_events"

print(f"⏳ Creating a clean baseline table: {demo_table} (Version 0)...")

# 2. Extract a small sample of our Gold table to act as our "Healthy" production data
healthy_data = spark.table("gold_predicted_buyers").limit(100)

# Write this to a new Delta table. Overwrite ensures we start completely fresh at Version 0.
healthy_data.write.format("delta").mode("overwrite").saveAsTable(demo_table)

# 3. Verify the Baseline
version_0_count = spark.table(demo_table).count()
print(f"   ✅ Version 0 created successfully with {version_0_count} rows.")
display(spark.table(demo_table).limit(5))

####Simulate a Bad Pipeline Append (Version 1)
Imagine your daily orchestration pipeline suffered a bug and accidentally ingested 500 rows of complete garbage data into your pristine table. Let's simulate that.

In [0]:
print("🚨 SIMULATING PIPELINE FAILURE: Appending corrupted data...")

# 1. Generate "Bad" Data with Explicit Casting
# We explicitly cast our generated data to match the standard Gold table schema
bad_data = (
    spark.range(0, 500)
    .withColumn("user_id", (F.col("id") * -1).cast("string")) # Cast to string to match standard IDs
    .withColumn("cart_count", F.lit(0).cast("integer"))
    .withColumn("view_count", F.lit(0).cast("integer"))
    .withColumn("purchase_intent_score", F.lit(999.0).cast("double")) # Impossible score
    .withColumn("total_events", F.lit(0).cast("integer"))
    .withColumn("predicted_to_buy", F.lit(0).cast("integer"))
    .drop("id")
)

# 2. Append the Bad Data to our Delta Table
bad_data.write.format("delta").mode("append").saveAsTable(demo_table)

# 3. Assess the Damage
version_1_count = spark.table(demo_table).count()
print(f"   ❌ Table is corrupted! Row count jumped from {version_0_count} to {version_1_count}.")

# Prove the garbage data is in "production"
display(
    spark.table(demo_table)
    .filter(F.col("purchase_intent_score") == 999.0)
    .limit(5)
)

####Audit History & Query Older Version (Time Travel)
Before we fix the table, we need to audit what happened and verify that our older, healthy data still exists on disk.

In [0]:
# ---------------------------------------------------------
# DELTA TIME TRAVEL & TRANSACTION LOG AUDIT
# ---------------------------------------------------------

# 1. Audit the Transaction History
print("🕵️‍♂️ Auditing Delta Lake Transaction Log...")
history_df = spark.sql(f"DESCRIBE HISTORY {demo_table}")

# The history table shows timestamps, operations (WRITE vs APPEND), and user metrics
display(history_df.select("version", "timestamp", "operation", "operationParameters"))

# 2. Time Travel Query
print("\n⏱️ Time Traveling to query Version 0 (Before the bad append)...")

# We use the DataFrame API with the "versionAsOf" option to look back in time
healthy_version_df = (
    spark.read.format("delta")
    .option("versionAsOf", 0) 
    .table(demo_table)
)

# Compare the counts to prove Time Travel works perfectly
current_count = spark.table(demo_table).count() # Reads the latest (Version 1)
past_count = healthy_version_df.count()         # Reads Version 0

print(f"   ➤ Current 'Production' rows (V1): {current_count}")
print(f"   ➤ Time-Traveled rows (V0): {past_count}")

# 💡 PRO TIP FOR INTERVIEWS:
# You can also time travel by timestamp instead of version number!
# Example: .option("timestampAsOf", "2026-03-02 03:00:00")

####The "3 AM Lifesaver" (Data Recovery)
We have proven Version 0 is safe. Now, let's execute a Delta RESTORE command to revert our production table back to that exact state, effectively deleting the pipeline mistake.

In [0]:
print(f"🚑 Initiating Data Recovery for {demo_table}...")

# 1. Restore the table to Version 0
# This is natively supported in Databricks SQL and is highly optimized
spark.sql(f"RESTORE TABLE {demo_table} TO VERSION AS OF 0")

print("   ✅ Restore complete!")

# 2. Verify the Fix
final_count = spark.table(demo_table).count()
print(f"   ➤ Final row count: {final_count} (Should match our baseline of {version_0_count})")

# Let's verify those 999.0 impossible scores are gone
bad_rows_remaining = spark.table(demo_table).filter(F.col("purchase_intent_score") == 999.0).count()
print(f"   ➤ Corrupted rows remaining: {bad_rows_remaining}")

# 3. Clean up the sandbox table
# Keep your Lakehouse tidy by dropping the demo table when done
spark.sql(f"DROP TABLE {demo_table}")
print("🧹 Sandbox demo table dropped. Day 11 complete!")

####Key Learnings & Interview Talking Points
If an interviewer asks you about data reliability, handling pipeline failures, or Delta Lake internals, use these critical points:

* **The Delta Transaction Log (`_delta_log`)**: "I understand that Delta Lake achieves ACID compliance by maintaining a sequential JSON transaction log alongside the underlying Parquet files. When running `DESCRIBE HISTORY`, I am querying this log to audit every single insert, update, delete, or schema change applied to the table."

* **Zero-Copy Time Travel**: "I actively use PySpark's `.option("versionAsOf", N)` to perform Time Travel. This feature is a massive cost-saver because it doesn't duplicate data. It simply reads the specific snapshot of Parquet files that were active at that point in time, allowing Data Scientists to reproducibly train ML models on historical data snapshots."

* **Instant Rollbacks via `RESTORE`**: "In legacy Data Warehouses, rolling back from a corrupted pipeline append requires loading massive physical backups. In a Lakehouse architecture, I use the `RESTORE TABLE ... TO VERSION` AS OF command. This instantly updates the transaction log to ignore the corrupted Parquet files, resulting in a near-instantaneous recovery with zero downtime for downstream consumers."

* **Data Auditing**: "When a data quality issue arises in production, I don't just fix it; I use `DESCRIBE HISTORY` to pinpoint exactly when the anomaly was introduced and by which cluster or pipeline job, ensuring rapid root-cause analysis